# LSTM

Using the same preprocessed processed_data_V3

Unlike the RF (which flattens each clip into summary statistics) or the CNN (which looks at local temporal windows with filters), the LSTM processes frames one at a time and maintains a hidden state across the full sequence. This allows it to model long-range dependencies between frames.

In [2]:
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

DATA_DIR    = './processed_data_New1'
RANDOM_SEED = 80
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# If this prints a GPU, training will be ~15-20x faster than CPU. If it's empty, CPU is fine.
print('GPUs visible to TF:', tf.config.list_physical_devices('GPU'))

I0000 00:00:1776571204.268063    5713 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776571204.313725    5713 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776571205.261774    5713 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


GPUs visible to TF: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# load data 
dev = np.load(os.path.join(DATA_DIR, 'dev.npz'))
test = np.load(os.path.join(DATA_DIR, 'test.npz'))
folds = np.load(os.path.join(DATA_DIR, 'cv_folds.npz'))
classes = np.load(os.path.join(DATA_DIR, 'classes.npy'), allow_pickle=True)

X_dev, y_dev, groups_dev = dev['X'], dev['y'], dev['groups']
X_test, y_test = test['X'], test['y']
num_classes = len(classes)
input_shape = X_dev.shape[1:]

print(f'Dev:  {X_dev.shape}  ({len(np.unique(groups_dev))} participants)')
print(f'Test: {X_test.shape}')
print(f'Classes: {num_classes}')
print(f'Random chance: {100/num_classes:.1f}%')

Dev:  (16007, 30, 126)  (18 participants)
Test: (2841, 30, 126)
Classes: 50
Random chance: 2.0%


In [5]:
# ── Model builder ─────────────────────────────────────────────────────────────
# Two-layer bidirectional LSTM. 'Bidirectional' means the model reads the
# sequence forward AND backward and combines both — useful because an ASL sign
# is often most recognizable from its end pose looking back, not only from
# the start looking forward.

def build_lstm(input_shape=input_shape, num_classes=50,
               lstm_units=(128, 64), dropout=0.4, recurrent_dropout=0.0,
               dense_size=64, bidirectional=True):

    def lstm_layer(units, return_sequences):
        layer = LSTM(units, return_sequences=return_sequences, recurrent_dropout=recurrent_dropout)
        return Bidirectional(layer) if bidirectional else layer

    model = Sequential([
        Input(shape=input_shape),

        # First LSTM layer returns a sequence so the second can process it.
        lstm_layer(lstm_units[0], return_sequences=True),
        Dropout(dropout),

        # Second LSTM layer returns only the final hidden state — a summary
        # of the whole clip.
        lstm_layer(lstm_units[1], return_sequences=False),
        Dropout(dropout),

        Dense(dense_size, activation='relu'),
        BatchNormalization(),
        Dropout(dropout),

        Dense(num_classes, activation='softmax'),
    ])

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model
build_lstm(input_shape=input_shape, num_classes=num_classes).summary()

I0000 00:00:1776571329.013328    5713 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5590 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 30, 256)        │       261,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         3,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 437,234 (1.67 MB)

 Trainable params: 437,106 (1.67 MB)

 Non-trainable params: 128 (512.00 B)

In [7]:
# ── 5-fold GroupKFold CV with baseline config ─────────────────────────────────
# Time per fold: ~5-10 min on GPU, ~30-60 min on CPU.
# Budget: ~30-60 min on GPU, ~3-5 hours on CPU.

n_folds = 5
fold_accs = []

for fold in range(n_folds):
    print(f'\n=== Fold {fold+1}/{n_folds} ===')
    tr = folds[f'fold_{fold}_train']
    va = folds[f'fold_{fold}_val']

    X_tr, y_tr = X_dev[tr], y_dev[tr]
    X_va, y_va = X_dev[va], y_dev[va]
    y_tr_cat = to_categorical(y_tr, num_classes)
    y_va_cat = to_categorical(y_va, num_classes)

    model = build_lstm(input_shape = input_shape, num_classes=num_classes)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5),
    ]
    model.fit(X_tr, y_tr_cat,
              validation_data=(X_va, y_va_cat),
              epochs=80, batch_size=128,
              callbacks=callbacks, verbose=2)

    _, acc = model.evaluate(X_va, y_va_cat, verbose=0)
    fold_accs.append(acc)
    print(f'Fold {fold+1} accuracy: {acc*100:.2f}%')

print(f'\nMean CV: {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%')


=== Fold 1/5 ===
Epoch 1/80
391/391 - 17s - 43ms/step - accuracy: 0.0195 - loss: 4.1728 - val_accuracy: 0.0165 - val_loss: 3.9170 - learning_rate: 0.0010
Epoch 2/80
391/391 - 15s - 38ms/step - accuracy: 0.0218 - loss: 3.9754 - val_accuracy: 0.0182 - val_loss: 3.9116 - learning_rate: 0.0010
Epoch 3/80
391/391 - 14s - 36ms/step - accuracy: 0.0323 - loss: 3.8430 - val_accuracy: 0.0338 - val_loss: 3.7646 - learning_rate: 0.0010
Epoch 4/80
391/391 - 15s - 37ms/step - accuracy: 0.0394 - loss: 3.7230 - val_accuracy: 0.0651 - val_loss: 3.6405 - learning_rate: 0.0010
Epoch 5/80
391/391 - 15s - 38ms/step - accuracy: 0.0640 - loss: 3.5890 - val_accuracy: 0.0921 - val_loss: 3.5096 - learning_rate: 0.0010
Epoch 6/80
391/391 - 15s - 39ms/step - accuracy: 0.0854 - loss: 3.4565 - val_accuracy: 0.1032 - val_loss: 3.4970 - learning_rate: 0.0010
Epoch 7/80
391/391 - 14s - 35ms/step - accuracy: 0.1054 - loss: 3.3539 - val_accuracy: 0.1165 - val_loss: 3.3594 - learning_rate: 0.0010
Epoch 8/80
391/391 - 15

In [ ]:
# ── Hyperparameter tuning ─────────────────────────────────────────────────────
# Each config = 5 folds. Pick a small grid unless you have GPU time to burn.
# Start with 3-5 configs; add more only if you see promising trends.

param_grid = [
    # Baseline
    {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': True},

    # Try smaller — might help if overfitting
    {'lstm_units': (64, 32),  'dropout': 0.4, 'bidirectional': True},

    # More dropout
    {'lstm_units': (128, 64), 'dropout': 0.5, 'bidirectional': True},

    # Unidirectional comparison
    {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': False},
]

tuning_results = []

for params in param_grid:
    print(f'\n\n>>> Config: {params}')
    accs = []
    for fold in range(n_folds):
        tr = folds[f'fold_{fold}_train']
        va = folds[f'fold_{fold}_val']
        y_tr_cat = to_categorical(y_dev[tr], num_classes)
        y_va_cat = to_categorical(y_dev[va], num_classes)

        model = build_lstm(input_shape = input_shape, num_classes=num_classes, **params)
        model.fit(X_dev[tr], y_tr_cat,
                  validation_data=(X_dev[va], y_va_cat),
                  epochs=60, batch_size=128,
                  callbacks=[
                      EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
                      ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5),
                  ],
                  verbose=0)
        _, acc = model.evaluate(X_dev[va], y_va_cat, verbose=0)
        accs.append(acc)
        print(f'  Fold {fold+1}: {acc*100:.2f}%')
    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    tuning_results.append((params, mean_acc, std_acc))
    print(f'  => Mean: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%')

best_params, best_acc, _ = max(tuning_results, key=lambda r: r[1])
print(f'\n\n=== Best config: {best_params} @ {best_acc*100:.2f}% ===')



>>> Config: {'lstm_units': (128, 64), 'dropout': 0.4, 'bidirectional': True}


In [ ]:
# ── Final model: train on full dev set with best params, evaluate on test ─────
# Use fold 0's val split as early-stopping watchdog, remaining dev as train.
# Keeps the final model selection honest: nothing touches the held-out test.

val_slice = folds['fold_0_val']
train_slice = np.setdiff1d(np.arange(len(X_dev)), val_slice)

y_dev_cat  = to_categorical(y_dev,  num_classes)
y_test_cat = to_categorical(y_test, num_classes)

final_model = build_lstm(input_sshape = input_shape, num_classes=num_classes, **best_params)
final_model.fit(
    X_dev[train_slice], y_dev_cat[train_slice],
    validation_data=(X_dev[val_slice], y_dev_cat[val_slice]),
    epochs=80, batch_size=128,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5),
    ],
    verbose=2
)

_, test_acc = final_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'\nHeld-out test accuracy: {test_acc*100:.2f}%')
print(f'Random chance:          {100/num_classes:.1f}%')

In [ ]:
# ── Per-class report (professor's suggestion #2) ──────────────────────────────
y_pred = final_model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=classes, zero_division=0))

In [ ]:
# ── Confusion matrix (professor's suggestion #3) ──────────────────────────────
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, xticklabels=classes, yticklabels=classes, cmap='Blues',
            square=True, cbar_kws={'label': 'Count'})
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'LSTM Confusion Matrix — Test Accuracy {test_acc*100:.1f}%')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('lstm_confusion_matrix.png', dpi=120)
plt.show()

cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
print('\nTop 10 confused pairs:')
for idx in np.argsort(cm_no_diag.ravel())[-10:][::-1]:
    i, j = np.unravel_index(idx, cm_no_diag.shape)
    if cm_no_diag[i, j] > 0:
        print(f'  {classes[i]:>12} -> {classes[j]:<12} : {cm_no_diag[i, j]} times')